# Who Wants to be a PoliMillionaire? — chatbot investigation

**Course:** Natural Language Processing, Politecnico di Milano, AY 2025/26  
**Due:** 2 June 2026, 23:00 (WeBeep)

| Name | Email | GitHub | PoliMillionaire user |
|---|---|---|---|
| Leon Fuß | leon.fuss@icloud.com | leonfuss | leonfuss |
| Antoine Gaborieau | _ | AntoineGaborieau | _ |
| Aleksa Pulai | _ | aleksapulai | _ |
| Luco _ | _ | LucBruc | _ |
| _ | _ | _ | _ |

**Video:** <link to be added before submission>

**Coding assistants used:** _Edit before submission. Suggested template: During this project we used <tool name(s)> for scaffolding boilerplate, generating docstrings, and debugging stack traces. The model architectures, evaluation methodology, and final analyses were designed by us; we reviewed and edited every line of generated code before committing it. The assignment was not given as a whole to any LLM._

## Introduction

Who Wants to be a PoliMillionaire? is a four-category multiple-choice quiz game served via a REST API. Each game presents up to 15 questions with a 30-second server-side timer per question; a wrong answer ends the game. The four competitions are Entertainment (id 0), Ancient History and Politics (1), Science and Nature (2), and Maths (3).

We investigated four bot strategies in increasing order of complexity: a zero-shot LLM baseline, a calculator-augmented ReAct loop (calc_react) for numeric reasoning, a retrieval-augmented variant that pulls MATH-corpus exemplars before reasoning (rag_calc_react), and a per-competition router (auto) that dispatches to Wikipedia-RAG for categories 0–2 and to rag_calc_react for category 3. All strategies share the same `AnswerDecision` interface and write to the same SQLite log, so offline replay is a fair apples-to-apples comparison.

This notebook reports accuracy by (strategy, model, competition), prompt sensitivity, latency, and an ablation over the retrieval components of wiki_rag. The analysis runs offline against the logged question corpus; the live-play cells at the end require valid server credentials and are not meant to run in CI.

## Setup

The cell below is Colab-specific. It clones the repo (or pulls latest main if already present), installs the heavy ML dependencies from `requirements-colab.txt`, and mounts Drive so the shared question log is accessible. Set `GH_TOKEN`, `POLIMILLIONAIRE_API_URL`, `POLIMILLIONAIRE_USER`, and `POLIMILLIONAIRE_PASSWORD` in Colab Secrets before running.

In [ ]:
# --- Colab-only bootstrap: skip this cell when running locally ---
import os
import sys

from google.colab import drive, userdata

gh_token = userdata.get("GH_TOKEN")
repo_dir = "/content/polimillionaire"

if os.path.isdir(repo_dir):
    os.chdir(repo_dir)
    !git checkout -q main && git pull --ff-only origin main
else:
    !git clone https://{gh_token}@github.com/leonfuss/polimillionaire.git $repo_dir
    os.chdir(repo_dir)

# CUDA wheel index for llama-cpp-python's pre-built T4 wheel.
!pip install -q -r requirements-colab.txt \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
    && pip install -q -e . --no-deps

if "/content/polimillionaire/src" not in sys.path:
    sys.path.insert(0, "/content/polimillionaire/src")

# Mount Drive and point the DB path at the shared question log.
drive.mount("/content/drive")
INDEX_ROOT = "/content/drive/MyDrive/PoliMillionaire"
os.environ.setdefault(
    "POLIMILLIONAIRE_DB_PATH",
    f"{INDEX_ROOT}/questions.sqlite",
)

# Symlink Drive's pre-built FAISS/BM25 index into the repo so the factory
# can find it at data/index/ without duplicating the ~1 GB onto the VM.
index_link = "/content/polimillionaire/data/index"
if not os.path.exists(index_link):
    os.makedirs("/content/polimillionaire/data", exist_ok=True)
    os.symlink(f"{INDEX_ROOT}/index", index_link)

print("DB path:", os.environ["POLIMILLIONAIRE_DB_PATH"])

## Load shared resources

Load the LLM and set the DB path. Adjust `model_alias` to switch between available models; the factory caches the loaded weights so repeated calls in the same session are free.

In [ ]:
import os
from pathlib import Path

from polimillionaire import load_llm

llm = load_llm("qwen3-8b")
DB_PATH = Path(os.environ.get("POLIMILLIONAIRE_DB_PATH", "data/questions.sqlite"))
print("DB path:", DB_PATH, "| exists:", DB_PATH.exists())

## Manual baseline

We bootstrapped the labelled corpus by playing manually before LLM strategies were ready. Each question seen in a live game — together with the correct answer revealed by the server — is logged to the shared SQLite DB. This gives us a set of labelled questions we can replay any strategy against offline without touching the live API or the 30-second timer.

The corpus currently contains around 234 labelled questions across the four competitions.

## Offline replay infrastructure

`replay_records` iterates over every labelled question in the DB, calls the strategy on each, and returns a flat list of `ReplayResult` objects. `to_polars` converts that list to a polars DataFrame. This is the evaluation loop used by all sections below.

In [ ]:
import polars as pl

from polimillionaire.eval.replay import replay_records
from polimillionaire.eval.results import to_polars
from polimillionaire.strategies import make_strategy

## Leaderboard

Replay several (strategy, model) combinations and collect per-question results into one DataFrame. Accuracy is the fraction of questions answered correctly; `n` is the number of labelled questions replayed. Uncomment entries in `STRATEGIES_TO_COMPARE` as more strategies stabilise.

In [ ]:
STRATEGIES_TO_COMPARE = [
    ("zero_shot", "qwen3-8b"),
    # ("calc_react", "qwen3-8b"),
    # ("auto", "qwen3-8b"),
]

frames = []
for strategy_name, model_alias in STRATEGIES_TO_COMPARE:
    _llm = load_llm(model_alias)
    strategy = make_strategy(strategy_name, _llm)
    records = replay_records(strategy, DB_PATH, show_progress=True)
    frames.append(to_polars(records))

df = pl.concat(frames) if frames else pl.DataFrame()

if not df.is_empty():
    df.group_by(["strategy_name", "model_name", "competition_id"]).agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("correct").count().alias("n"),
    ).sort(["strategy_name", "competition_id"])

## Accuracy by competition

Pivot the leaderboard DataFrame so each row is a (strategy, model) pair and each column is a competition. Makes it easy to spot which categories benefit most from retrieval augmentation.

In [ ]:
if not df.is_empty():
    df.group_by(["strategy_name", "model_name", "competition_id"]).agg(
        pl.col("correct").mean().alias("accuracy"),
    ).pivot(
        on="competition_id",
        index=["strategy_name", "model_name"],
        values="accuracy",
    ).sort("strategy_name")

## Latency vs accuracy

Each strategy incurs different per-question latency. Plot mean latency against accuracy to see whether the more expensive strategies (rag_calc_react, auto) justify their wall-clock cost.

In [ ]:
import matplotlib.pyplot as plt
from tueplots import bundles

plt.rcParams.update(bundles.icml2024())

if not df.is_empty():
    agg = df.group_by(["strategy_name", "model_name"]).agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("latency_ms").mean().alias("mean_latency_ms"),
    )

    fig, ax = plt.subplots()
    for row in agg.iter_rows(named=True):
        ax.scatter(row["mean_latency_ms"], row["accuracy"], label=row["strategy_name"])
        ax.annotate(row["strategy_name"], (row["mean_latency_ms"], row["accuracy"]), fontsize=6)
    ax.set_xlabel("mean latency (ms)")
    ax.set_ylabel("accuracy")
    ax.legend(fontsize=6)
    plt.tight_layout()
    plt.show()

## Ablation: wiki_rag retrieval components

The wiki_rag strategy has three optional retrieval components: dense (FAISS), sparse (BM25), and a cross-encoder reranker. This cell sweeps over the on/off combinations for a single competition to isolate which component drives most of the accuracy gain. When the index is absent the strategy degrades to zero_shot automatically.

In [ ]:
ABLATION_COMPETITION_ID = 2  # Science and Nature; change as needed

ablation_configs = [
    ({"use_dense": True, "use_sparse": True, "use_reranker": True}, "dense+sparse+rerank"),
    ({"use_dense": True, "use_sparse": True, "use_reranker": False}, "dense+sparse"),
    ({"use_dense": True, "use_sparse": False, "use_reranker": False}, "dense-only"),
    ({"use_dense": False, "use_sparse": True, "use_reranker": False}, "sparse-only"),
]

ablation_frames = []
for cfg, label in ablation_configs:
    try:
        # strict=True makes the factory raise if the wiki index is missing,
        # so we don't silently compare four identical zero-shot runs.
        s = make_strategy(
            "wiki_rag",
            llm,
            competition_id=ABLATION_COMPETITION_ID,
            strict=True,
            **cfg,
        )
    except FileNotFoundError as e:
        print(f"skipping ablation: {e}")
        break
    records = replay_records(
        s,
        DB_PATH,
        competition_id=ABLATION_COMPETITION_ID,
        show_progress=True,
    )
    frame = to_polars(records).with_columns(pl.lit(label).alias("ablation"))
    ablation_frames.append(frame)

ablation_df = pl.concat(ablation_frames) if ablation_frames else pl.DataFrame()
if not ablation_df.is_empty():
    ablation_df.group_by("ablation").agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("correct").count().alias("n"),
    ).sort("accuracy", descending=True)

## Prompt sensitivity

Each strategy records the `prompt_version` used at generation time. This cell compares accuracy across versions for a fixed (strategy, model) pair — useful for tracking whether a prompt edit helped or hurt.

In [ ]:
PROMPT_STRATEGY = "zero_shot"
PROMPT_VERSIONS = ["v1"]  # extend as new versions land

prompt_frames = []
for pv in PROMPT_VERSIONS:
    s = make_strategy(PROMPT_STRATEGY, llm, prompt_version=pv)
    records = replay_records(s, DB_PATH, show_progress=True)
    prompt_frames.append(to_polars(records))

prompt_df = pl.concat(prompt_frames) if prompt_frames else pl.DataFrame()

if not prompt_df.is_empty():
    prompt_df.group_by("prompt_version").agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("correct").count().alias("n"),
    ).sort("prompt_version")

## Model comparison

Hold the strategy fixed and sweep across model aliases. This isolates the contribution of model size and quantisation from prompt and retrieval choices.

In [ ]:
MODEL_STRATEGY = "zero_shot"
MODELS_TO_COMPARE = ["qwen3-8b"]  # extend with other entries from MODELS

model_frames = []
for alias in MODELS_TO_COMPARE:
    _llm = load_llm(alias)
    s = make_strategy(MODEL_STRATEGY, _llm)
    records = replay_records(s, DB_PATH, show_progress=True)
    model_frames.append(to_polars(records))

model_df = pl.concat(model_frames) if model_frames else pl.DataFrame()

if not model_df.is_empty():
    model_df.group_by(["model_name", "strategy_name"]).agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("correct").count().alias("n"),
    ).sort("accuracy", descending=True)

## Qualitative trace

Pick a specific question from the log and show the strategy's full rationale. This is useful for diagnosing systematic errors — e.g. the model consistently mis-classifying numeric questions, or the retriever surfacing an irrelevant passage.

In [ ]:
import json
import sqlite3

from polimillionaire._vendor.millionaire_client.models import Option, Question
from polimillionaire.strategies import ZeroShotStrategy
from polimillionaire.strategies.base import Context

TRACE_QUESTION_ID = None  # set to an integer question_id from the DB

if TRACE_QUESTION_ID is not None and DB_PATH.exists():
    with sqlite3.connect(DB_PATH) as con:
        con.row_factory = sqlite3.Row
        row = con.execute(
            """
            SELECT question_id, question_text, options_json, level, competition_id,
                   correct_option_id_if_known
            FROM predictions
            WHERE question_id = ? AND correct_option_id_if_known IS NOT NULL
            LIMIT 1
            """,
            (TRACE_QUESTION_ID,),
        ).fetchone()

    if row:
        options = [Option(**o) for o in json.loads(row["options_json"])]
        q = Question(
            id=row["question_id"],
            text=row["question_text"],
            options=options,
            level=row["level"],
        )
        ctx = Context(competition_id=row["competition_id"], level=row["level"])
        trace_strategy = ZeroShotStrategy(llm)  # swap for any strategy
        decision = trace_strategy(q, ctx)

        print(f"Q: {q.text}")
        for opt in q.options:
            marker = "*" if opt.id == row["correct_option_id_if_known"] else " "
            chosen = "->" if opt.id == decision.option_id else "  "
            print(f"  {chosen}[{marker}] [{opt.id}] {opt.text}")
        print(f"\nrationale: {decision.rationale}")
        print(f"confidence: {decision.confidence}  latency: {decision.latency_ms} ms")
    else:
        print(f"question_id {TRACE_QUESTION_ID} not found in DB")
else:
    print("Set TRACE_QUESTION_ID to an integer to run this cell.")

## Live play

Run a real game against the server using the deployed strategy. Requires valid credentials in Colab Secrets (or a `.env` file locally). This cell will not run in CI — it needs a live server connection and burns one of the team's play slots.

In [ ]:
# Set RUN_LIVE = True before executing this cell to actually play a game.
# Without the guard, "Run All" would burn a play slot and write a row to
# the shared SQLite log.
RUN_LIVE = False

if RUN_LIVE:
    from polimillionaire import make_client
    from polimillionaire.play import auto_play_loop

    client = make_client()
    live_strategy = make_strategy("auto", llm, competition_id=2)
    auto_play_loop(client, competition_id=2, strategy=live_strategy, max_games=1)
else:
    print("RUN_LIVE is False -- skipping live play")

## Conclusions

- _placeholder: overall accuracy of the best strategy vs zero-shot baseline_
- _placeholder: which competition benefited most from retrieval augmentation_
- _placeholder: latency budget — which strategies are viable under the 30 s timer_
- _placeholder: prompt sensitivity findings_
- _placeholder: failure modes observed in the qualitative trace_